# 01 · EDA — Amazon Movies & TV Reviews (PySpark)

Dataset: Amazon Reviews 2023 · Movies & TV · 6.5M users · 17.3M ratings · up to Sep 2023  
Source: https://amazon-reviews-2023.github.io/

In [ ]:
# ── Update to your Databricks volume path ─────────────────────────────────────
DATA_DIR = "/Volumes/workspace/ds_amazonrec/data"
# ─────────────────────────────────────────────────────────────────────────────

## 1 · Load Data

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, FloatType, LongType, BooleanType

# Reviews were split into parts for upload — Spark merges them automatically
raw_reviews = spark.read.json(f"{DATA_DIR}/reviews_part*.jsonl")

# Explicit schema avoids duplicate-column errors in the metadata JSON
meta_schema = StructType([
    StructField("parent_asin",    StringType(), True),
    StructField("title",          StringType(), True),
    StructField("average_rating", FloatType(),  True),
    StructField("price",          FloatType(),  True),
])
raw_meta = spark.read.schema(meta_schema).json(f"{DATA_DIR}/meta_Movies_and_TV.jsonl")

reviews = raw_reviews.select(
    F.col("user_id"),
    F.col("parent_asin").alias("product_id"),
    F.col("rating").cast("float"),
    F.col("timestamp"),
    F.col("verified_purchase"),
)

products = raw_meta.select(
    F.col("parent_asin").alias("product_id"),
    F.col("title").alias("name"),
    F.col("average_rating"),
    F.col("price"),
).dropDuplicates(["product_id"])

reviews.printSchema()
reviews.show(5, truncate=False)

## 2 · Basic Stats

In [ ]:
reviews.select(
    F.count("*").alias("total_reviews"),
    F.countDistinct("user_id").alias("unique_users"),
    F.countDistinct("product_id").alias("unique_products"),
    F.round(F.mean("rating"), 3).alias("avg_rating"),
    F.min("rating").alias("min_rating"),
    F.max("rating").alias("max_rating"),
).show()

In [ ]:
reviews.groupBy("verified_purchase").count() \
    .withColumn("pct", F.round(F.col("count") / reviews.count() * 100, 1)) \
    .orderBy("verified_purchase") \
    .show()

## 3 · Rating Distribution

In [ ]:
import matplotlib.pyplot as plt

rating_dist = (
    reviews.groupBy("rating")
    .count()
    .orderBy("rating")
    .toPandas()
)

plt.figure(figsize=(8, 4))
plt.bar(rating_dist["rating"], rating_dist["count"] / 1e6,
        color="steelblue", edgecolor="white", width=0.6)
plt.title("Rating Distribution — Movies & TV")
plt.xlabel("Stars")
plt.ylabel("Reviews (millions)")
plt.tight_layout()
plt.show()

rating_dist["pct"] = (rating_dist["count"] / rating_dist["count"].sum() * 100).round(1)
print(rating_dist)

## 4 · Activity Distribution

In [ ]:
user_counts = reviews.groupBy("user_id").count()
user_counts.select(
    F.round(F.mean("count"), 2).alias("avg_reviews_per_user"),
    F.percentile_approx("count", 0.5).alias("median"),
    F.max("count").alias("max"),
).show()

product_counts = reviews.groupBy("product_id").count()
product_counts.select(
    F.round(F.mean("count"), 2).alias("avg_reviews_per_product"),
    F.percentile_approx("count", 0.5).alias("median"),
    F.max("count").alias("max"),
).show()

## 5 · Top 20 Most-Reviewed Products

In [ ]:
import seaborn as sns

top_products = (
    reviews
    .groupBy("product_id")
    .agg(
        F.count("*").alias("review_count"),
        F.round(F.mean("rating"), 2).alias("avg_rating")
    )
    .join(products, "product_id", "left")
    .orderBy(F.col("review_count").desc())
    .limit(20)
    .toPandas()
)

plt.figure(figsize=(10, 7))
sns.barplot(data=top_products, x="review_count", y="name", palette="Blues_r")
plt.title("Top 20 Most-Reviewed Movies & TV Products")
plt.xlabel("Number of reviews")
plt.ylabel("")
plt.tight_layout()
plt.show()

## 6 · Reviews Over Time

In [ ]:
reviews_by_year = (
    reviews
    .withColumn("year", F.year(F.to_timestamp(F.col("timestamp") / 1000)))
    .groupBy("year")
    .count()
    .orderBy("year")
    .filter(F.col("year") >= 2000)
    .toPandas()
)

plt.figure(figsize=(11, 4))
plt.bar(reviews_by_year["year"], reviews_by_year["count"] / 1e6,
        color="tomato", edgecolor="white")
plt.title("Reviews per Year (Movies & TV)")
plt.xlabel("Year")
plt.ylabel("Reviews (millions)")
plt.tight_layout()
plt.show()

## 7 · 5-Core Filtering

Keep only users with ≥5 reviews and products with ≥5 reviews.

In [ ]:
def apply_k_core(df, k=5, max_iter=10):
    for i in range(max_iter):
        n_before = df.count()
        product_counts = df.groupBy("product_id").count()
        df = df.join(product_counts.filter(F.col("count") >= k).select("product_id"), "product_id")
        user_counts = df.groupBy("user_id").count()
        df = df.join(user_counts.filter(F.col("count") >= k).select("user_id"), "user_id")
        n_after = df.count()
        print(f"Iteration {i+1}: {n_before:,} → {n_after:,} rows")
        if n_before == n_after:
            break
    return df

reviews_5core = apply_k_core(reviews.select("user_id", "product_id", "rating", "timestamp"))

print()
reviews_5core.select(
    F.count("*").alias("interactions"),
    F.countDistinct("user_id").alias("users"),
    F.countDistinct("product_id").alias("products"),
).show()

## 8 · Matrix Sparsity

In [ ]:
stats = reviews_5core.select(
    F.count("*").alias("nnz"),
    F.countDistinct("user_id").alias("n_users"),
    F.countDistinct("product_id").alias("n_products")
).collect()[0]

sparsity = 1 - stats["nnz"] / (stats["n_users"] * stats["n_products"])
print(f"Matrix : {stats['n_users']:,} users × {stats['n_products']:,} products")
print(f"Ratings: {stats['nnz']:,}")
print(f"Density: {1-sparsity:.4%}  (Sparsity: {sparsity:.2%})")